# 09 Overtake Analysis

Overtaking captures racecraft, circuit passability, and traffic dynamics. This notebook analyzes who creates overtakes, who loses positions, where overtaking clusters, and whether overtaking signal aligns with finish position.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "09_overtake_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def add_event_type(sessions: pd.DataFrame) -> pd.DataFrame:
    sessions = sessions.copy()
    sessions["event_type"] = np.where(
        sessions["session_name"].astype(str).str.lower().eq("sprint"),
        "SPRINT_RACE",
        "GRAND_PRIX_RACE",
    )
    return sessions

def save_fig(fig, name: str):
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")
    fig.show()

print("=" * 72)
print(f"SILVER STRATEGY EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER STRATEGY EDA - 09_overtake_analysis
Start time: 2026-06-02 02:14:11.574967
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
overtakes = pd.read_parquet(CLEANED_DATA_PATH / "overtakes.parquet")
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
sessions = add_event_type(pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet"))
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")

for column in ["overtaking_driver_number", "overtaken_driver_number", "position", "session_key"]:
    if column in overtakes.columns:
        overtakes[column] = pd.to_numeric(overtakes[column], errors="coerce")
overtakes["date"] = pd.to_datetime(overtakes["date"], errors="coerce", utc=True)
overtakes = overtakes.merge(sessions[["session_key", "year", "event_type", "circuit_short_name", "date_start", "date_end"]], on="session_key", how="left")
for column in ["date_start", "date_end"]:
    overtakes[column] = pd.to_datetime(overtakes[column], errors="coerce", utc=True)
duration_seconds = (overtakes["date_end"] - overtakes["date_start"]).dt.total_seconds()
overtakes["race_phase_pct"] = ((overtakes["date"] - overtakes["date_start"]).dt.total_seconds() / duration_seconds * 100).clip(0, 100)
overtakes["race_phase"] = pd.cut(overtakes["race_phase_pct"], bins=[-1, 25, 50, 75, 101], labels=["Start", "Early-Mid", "Late-Mid", "Final"])

coverage = pd.DataFrame([{
    "rows": len(overtakes),
    "sessions": overtakes["session_key"].nunique(),
    "circuits": overtakes["circuit_short_name"].nunique(),
    "drivers_making_overtakes": overtakes["overtaking_driver_number"].nunique(),
}])
coverage.to_csv(OUTPUT_TABLES / "overtake_coverage.csv", index=False)
display(coverage)

,rows,sessions,circuits,drivers_making_overtakes
0,13985,68,24,31


## 1. Overtake Volume and Timing

Overtake count by session tells us which races are strategically alive versus processional. Race phase helps identify whether passing happens mostly during opening chaos, tyre-offset windows, or late-race recovery.

In [3]:
session_overtakes = overtakes.groupby(["session_key", "year", "event_type", "circuit_short_name"], as_index=False).size().rename(columns={"size": "overtakes"})
session_overtakes.to_csv(OUTPUT_TABLES / "session_overtake_counts.csv", index=False)
display(session_overtakes.sort_values("overtakes", ascending=False).head(20))

fig = px.histogram(session_overtakes, x="overtakes", color="event_type", nbins=30, title="Overtakes per Session Distribution")
save_fig(fig, "overtakes_per_session")

,session_key,year,event_type,circuit_short_name,overtakes
56,10014,2025,GRAND_PRIX_RACE,Sakhir,642
0,9472,2024,GRAND_PRIX_RACE,Sakhir,413
36,9869,2025,GRAND_PRIX_RACE,Interlagos,367
31,9839,2025,GRAND_PRIX_RACE,Yas Marina Circuit,342
9,9539,2024,GRAND_PRIX_RACE,Catalunya,334
24,9644,2024,GRAND_PRIX_RACE,Las Vegas,332
37,9877,2025,GRAND_PRIX_RACE,Mexico City,320
15,9582,2024,GRAND_PRIX_RACE,Zandvoort,311
29,9673,2024,GRAND_PRIX_RACE,Shanghai,310
60,11234,2026,GRAND_PRIX_RACE,Melbourne,299


In [4]:
phase_summary = overtakes.groupby(["event_type", "race_phase"], observed=True, as_index=False).size().rename(columns={"size": "overtakes"})
phase_summary.to_csv(OUTPUT_TABLES / "overtake_phase_summary.csv", index=False)
display(phase_summary)

fig = px.bar(
    phase_summary,
    x="race_phase",
    y="overtakes",
    color="event_type",
    barmode="group",
    title="Overtake Timing by Race Phase",
)
save_fig(fig, "overtake_timing_phase")

,event_type,race_phase,overtakes
0,GRAND_PRIX_RACE,Start,5736
1,GRAND_PRIX_RACE,Early-Mid,4146
2,GRAND_PRIX_RACE,Late-Mid,1998
3,GRAND_PRIX_RACE,Final,793
4,SPRINT_RACE,Start,863
5,SPRINT_RACE,Early-Mid,191
6,SPRINT_RACE,Late-Mid,70
7,SPRINT_RACE,Final,188


## 2. Driver Racecraft Balance

Net overtakes separate aggressive progress from vulnerability in traffic. The join uses session-aware driver identity to avoid name collisions across seasons and teams.

In [5]:
driver_dim = drivers[["session_key", "driver_number", "full_name", "team_name"]].drop_duplicates(["session_key", "driver_number"])
made = overtakes.groupby(["session_key", "overtaking_driver_number"]).size().reset_index(name="overtakes_made").rename(columns={"overtaking_driver_number": "driver_number"})
lost = overtakes.groupby(["session_key", "overtaken_driver_number"]).size().reset_index(name="overtaken_count").rename(columns={"overtaken_driver_number": "driver_number"})
driver_racecraft = made.merge(lost, on=["session_key", "driver_number"], how="outer").fillna(0)
driver_racecraft = driver_racecraft.merge(driver_dim, on=["session_key", "driver_number"], how="left")
driver_racecraft["driver_id"] = driver_racecraft["full_name"].fillna("Driver " + driver_racecraft["driver_number"].astype(int).astype(str))
driver_racecraft["net_overtakes"] = driver_racecraft["overtakes_made"] - driver_racecraft["overtaken_count"]
driver_summary = driver_racecraft.groupby("driver_id", as_index=False).agg(
    overtakes_made=("overtakes_made", "sum"),
    overtaken_count=("overtaken_count", "sum"),
    net_overtakes=("net_overtakes", "sum"),
    sessions=("session_key", "nunique"),
)
driver_summary["net_per_session"] = driver_summary["net_overtakes"] / driver_summary["sessions"]
driver_summary = driver_summary.sort_values("net_overtakes", ascending=False)
driver_summary.to_csv(OUTPUT_TABLES / "driver_overtake_balance.csv", index=False)
display(driver_summary.head(20))

,driver_id,overtakes_made,overtaken_count,net_overtakes,sessions,net_per_session
16,Lewis HAMILTON,671.0,571.0,100.0,66,1.515152
5,Esteban OCON,848.0,764.0,84.0,66,1.272727
21,Oliver BEARMAN,512.0,429.0,83.0,41,2.024390
17,Liam LAWSON,551.0,479.0,72.0,47,1.531915
27,ZHOU Guanyu,339.0,285.0,54.0,30,1.800000
12,Kevin MAGNUSSEN,361.0,325.0,36.0,27,1.333333
7,Franco COLAPINTO,417.0,394.0,23.0,41,0.560976
3,Charles LECLERC,546.0,523.0,23.0,66,0.348485
14,Lance STROLL,772.0,754.0,18.0,67,0.268657
24,Sergio PEREZ,376.0,360.0,16.0,38,0.421053


In [6]:
fig = px.bar(
    driver_summary.head(20).sort_values("net_overtakes"),
    x="net_overtakes",
    y="driver_id",
    orientation="h",
    color="net_per_session",
    title="Driver Net Overtake Balance",
)
save_fig(fig, "driver_net_overtakes")

## 3. Circuit Passability and Result Signal

Circuit ranking identifies tracks where traffic can be solved on track versus tracks where qualifying and pit strategy become more decisive.

In [7]:
circuit_overtakes = session_overtakes.groupby("circuit_short_name", as_index=False).agg(
    sessions=("session_key", "nunique"),
    total_overtakes=("overtakes", "sum"),
    avg_overtakes=("overtakes", "mean"),
).sort_values("avg_overtakes", ascending=False)
circuit_overtakes.to_csv(OUTPUT_TABLES / "circuit_overtake_ranking.csv", index=False)
display(circuit_overtakes)

fig = px.bar(
    circuit_overtakes,
    x="avg_overtakes",
    y="circuit_short_name",
    orientation="h",
    color="sessions",
    title="Circuit Overtaking Ranking",
)
save_fig(fig, "circuit_overtake_ranking")

,circuit_short_name,sessions,total_overtakes,avg_overtakes
15,Sakhir,2,1055,527.500000
2,Catalunya,2,619,309.500000
22,Yas Marina Circuit,2,584,292.000000
23,Zandvoort,2,548,274.000000
10,Mexico City,2,526,263.000000
3,Hungaroring,2,493,246.500000
7,Las Vegas,2,490,245.000000
9,Melbourne,3,682,227.333333
18,Singapore,2,441,220.500000
8,Lusail,4,855,213.750000


In [8]:
finish = session_result[["session_key", "driver_number", "position", "points"]].copy()
finish["finish_pos"] = pd.to_numeric(finish["position"], errors="coerce")
overtake_finish = driver_racecraft.merge(finish, on=["session_key", "driver_number"], how="left")
overtake_corr = overtake_finish[["overtakes_made", "overtaken_count", "net_overtakes", "finish_pos", "points"]].corr()
overtake_corr.to_csv(OUTPUT_TABLES / "overtake_finish_correlations.csv")
display(overtake_corr.round(3))

fig = px.imshow(
    overtake_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Overtaking vs Race Outcome Correlations",
)
save_fig(fig, "overtake_finish_correlation")

,overtakes_made,overtaken_count,net_overtakes,finish_pos,points
overtakes_made,1.000,0.817,0.317,0.160,-0.121
overtaken_count,0.817,1.000,-0.288,0.266,-0.229
net_overtakes,0.317,-0.288,1.000,-0.209,0.176
finish_pos,0.160,0.266,-0.209,1.000,-0.817
points,-0.121,-0.229,0.176,-0.817,1.000


## Final Overtake Report

In [9]:
top_driver = driver_summary.iloc[0].to_dict() if len(driver_summary) else {}
top_circuit = circuit_overtakes.iloc[0].to_dict() if len(circuit_overtakes) else {}
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "status": "PASS",
    "overtake_rows": int(len(overtakes)),
    "sessions": int(overtakes["session_key"].nunique()),
    "top_net_overtake_driver": top_driver.get("driver_id"),
    "top_net_overtakes": float(top_driver.get("net_overtakes", 0)),
    "highest_overtake_circuit": top_circuit.get("circuit_short_name"),
    "highest_avg_overtakes": float(top_circuit.get("avg_overtakes", 0)),
}
write_report("overtake_analysis", report)
write_insight(
    "Silver Overtake Analysis Insights",
    [
        f"Analyzed {report['overtake_rows']:,} overtake events across {report['sessions']} sessions.",
        f"Top net overtake driver: {report['top_net_overtake_driver']} ({report['top_net_overtakes']:.0f}).",
        f"Highest average overtake circuit: {report['highest_overtake_circuit']}.",
    ],
    [],
    [
        "Use net overtakes and circuit passability as Gold racecraft features.",
        "Normalize overtake volume by event type because Sprint and Grand Prix sessions have different opportunity lengths.",
        "Use race_phase_pct for future strategy timing features.",
    ],
)
(CHECKPOINTS / "silver_overtake_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '09_overtake_analysis', 'timestamp': '2026-06-02T02:14:13.276117', 'status': 'PASS', 'overtake_rows': 13985, 'sessions': 68, 'top_net_overtake_driver': 'Lewis HAMILTON', 'top_net_overtakes': 100.0, 'highest_overtake_circuit': 'Sakhir', 'highest_avg_overtakes': 527.5}
